# XGBoost accidentologie - 3 runs d'optimisation

Objectif:
- entrainer 3 runs XGBoost avec variation d'hyperparametres
- comparer les performances et selectionner le meilleur
- exporter modeles, predictions et metadonnees dans `out/xgb_experiments`

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


def find_project_root(marker: str = "out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    return Path.cwd()


ROOT = find_project_root("out")
DATA_PATH = ROOT / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
ARTIFACT_DIR = ROOT / "out" / "xgb_experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "grave"
SEED = 42
CV_SPLITS = 3
N_ITER = 20
THRESHOLD = 0.5
MAX_ROWS = None  # ex: 50000 pour aller plus vite

# Memes 15 features que les autres notebooks pour comparaison equitable
PRODUCT15_V2 = [
    "dep",
    "lum",
    "atm",
    "catr",
    "agg",
    "int",
    "circ",
    "col",
    "vma_bucket",
    "catv_family_4",
    "manv_mode",
    "driver_age_bucket",
    "choc_mode",
    "driver_trajet_family",
    "time_bucket",
]

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

In [ ]:
assert DATA_PATH.exists(), f"Fichier introuvable: {DATA_PATH}"

df = pd.read_csv(DATA_PATH, sep=";", low_memory=False, nrows=MAX_ROWS)
assert TARGET in df.columns, f"Colonne cible absente: {TARGET}"

missing_feats = [c for c in PRODUCT15_V2 if c not in df.columns]
assert not missing_feats, f"Colonnes manquantes dans le CSV: {missing_feats}"

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df[PRODUCT15_V2].copy()
y = df[TARGET].copy()

print("Shape:", X.shape)
print("Taux grave=1:", round(float(y.mean()), 4))
print("Features utilisees:", list(X.columns))

## Adaptations necessaires pour XGBoost

- Split train/val/test stratifie (60/20/20)
- **val** sert a optimiser le seuil, **test** sert a evaluer les metriques finales
- XGBoost travaille sur des entrees numeriques: one-hot encoding des colonnes categorielles
- imputation des valeurs manquantes
- prise en compte du desequilibre de classes via `scale_pos_weight`

In [ ]:
# Split 60/20/20 : train / val (seuil) / test (evaluation finale)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp,
)

# Toutes les features product15_v2 sont categorielles
cat_cols = PRODUCT15_V2[:]
num_cols = []

num_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

transformers = [("cat", cat_pipe, cat_cols)]
if num_cols:
    transformers.insert(0, ("num", num_pipe, num_cols))

preprocess = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight_base = float(neg / max(pos, 1))

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Categorielles:", len(cat_cols), "| Numeriques:", len(num_cols))
print("Class balance train -> pos:", pos, "neg:", neg, "scale_pos_weight:", round(scale_pos_weight_base, 3))

## Trois runs d'optimisation

Runs prevus:
1. `xgb_auc_opt` optimise `roc_auc`
2. `xgb_f1_opt` optimise `f1`
3. `xgb_recall_opt` optimise `recall`

In [ ]:
def evaluate_binary(y_true: pd.Series, proba: np.ndarray, threshold: float = THRESHOLD) -> dict[str, float]:
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
    }


def best_threshold_by_f1(y_true: pd.Series, proba: np.ndarray) -> tuple[float, dict[str, float]]:
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = []
    for t in thresholds:
        m = evaluate_binary(y_true, proba, threshold=float(t))
        rows.append(m)
    df_thr = pd.DataFrame(rows)
    best_idx = int(df_thr["f1"].idxmax())
    best = df_thr.loc[best_idx].to_dict()
    return float(best["threshold"]), {k: float(v) for k, v in best.items()}


def make_search(scoring: str, param_dist: dict) -> RandomizedSearchCV:
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )

    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)

    return RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring=scoring,
        n_jobs=-1,
        cv=cv,
        random_state=SEED,
        verbose=1,
        refit=True,
    )


spw_mid = round(scale_pos_weight_base, 3)
spw_light = round(scale_pos_weight_base * 0.75, 3)
spw_high = round(scale_pos_weight_base * 1.5, 3)
spw_very_high = round(scale_pos_weight_base * 2.5, 3)

PARAM_AUC = {
    "model__n_estimators": [300, 500, 800, 1200],
    "model__max_depth": [3, 4, 6, 8, 10],
    "model__learning_rate": [0.03, 0.05, 0.08, 0.12],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5, 8],
    "model__gamma": [0.0, 0.5, 1.0],
    "model__reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "model__reg_lambda": [1.0, 2.0, 5.0, 10.0],
    "model__scale_pos_weight": [spw_light, spw_mid, spw_high],
}

PARAM_F1 = {
    "model__n_estimators": [400, 700, 1000, 1400],
    "model__max_depth": [3, 5, 7, 9],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
    "model__subsample": [0.75, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 4, 6],
    "model__gamma": [0.0, 0.2, 0.6, 1.0],
    "model__reg_alpha": [0.0, 0.05, 0.2, 0.8],
    "model__reg_lambda": [1.0, 3.0, 7.0, 12.0],
    "model__scale_pos_weight": [spw_mid, spw_high],
}

PARAM_RECALL = {
    "model__n_estimators": [500, 900, 1300, 1700],
    "model__max_depth": [3, 4, 6, 8],
    "model__learning_rate": [0.015, 0.02, 0.03, 0.05],
    "model__subsample": [0.75, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 3, 5],
    "model__gamma": [0.0, 0.2, 0.4, 0.8],
    "model__reg_alpha": [0.0, 0.1, 0.4, 1.0],
    "model__reg_lambda": [1.0, 2.0, 5.0, 10.0],
    "model__scale_pos_weight": [spw_mid, spw_high, spw_very_high],
}

EXPERIMENTS = [
    ("xgb_auc_opt", "roc_auc", PARAM_AUC),
    ("xgb_f1_opt", "f1", PARAM_F1),
    ("xgb_recall_opt", "recall", PARAM_RECALL),
]

In [ ]:
trained = {}
rows = []

for run_name, scoring, param_dist in EXPERIMENTS:
    print(f"\n=== {run_name} | scoring={scoring} ===")

    search = make_search(scoring=scoring, param_dist=param_dist)
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # Seuil optimise sur val (pas sur test)
    proba_val = best_model.predict_proba(X_val)[:, 1]
    best_thr, _ = best_threshold_by_f1(y_val, proba_val)

    # Metriques finales sur test (jamais vu par le modele ni le seuil)
    proba_test = best_model.predict_proba(X_test)[:, 1]
    metrics_05 = evaluate_binary(y_test, proba_test, threshold=THRESHOLD)
    metrics_best = evaluate_binary(y_test, proba_test, threshold=best_thr)

    trained[run_name] = {
        "search": search,
        "model": best_model,
        "proba_test": proba_test,
        "metrics_05": metrics_05,
        "best_threshold": best_thr,
        "metrics_best": metrics_best,
    }

    rows.append(
        {
            "run_name": run_name,
            "optimized_for": scoring,
            "cv_best_score": float(search.best_score_),
            "threshold_05": float(THRESHOLD),
            "accuracy_05": metrics_05["accuracy"],
            "precision_05": metrics_05["precision"],
            "recall_05": metrics_05["recall"],
            "f1_05": metrics_05["f1"],
            "roc_auc_05": metrics_05["roc_auc"],
            "pr_auc_05": metrics_05["pr_auc"],
            "best_threshold_f1": best_thr,
            "precision_best": metrics_best["precision"],
            "recall_best": metrics_best["recall"],
            "f1_best": metrics_best["f1"],
            "best_params": search.best_params_,
        }
    )

results_df = pd.DataFrame(rows).sort_values(["f1_best", "roc_auc_05"], ascending=False).reset_index(drop=True)
results_df[["run_name", "optimized_for", "cv_best_score", "f1_05", "roc_auc_05", "f1_best", "best_threshold_f1"]]

In [ ]:
registry_rows = []

for row in rows:
    run_name = row["run_name"]
    payload = trained[run_name]

    model = payload["model"]
    proba_test = payload["proba_test"]
    best_thr = payload["best_threshold"]

    model_path = ARTIFACT_DIR / f"{run_name}.joblib"
    pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
    meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

    joblib.dump(model, model_path)

    pred_df = pd.DataFrame(
        {
            "y_true": y_test.to_numpy(),
            "proba": proba_test,
            "pred_05": (proba_test >= THRESHOLD).astype(int),
            "pred_best_f1": (proba_test >= best_thr).astype(int),
        }
    )
    pred_df.to_csv(pred_path, index=False)

    meta = {
        "run_name": run_name,
        "optimized_for": row["optimized_for"],
        "dataset": str(DATA_PATH),
        "target": TARGET,
        "seed": SEED,
        "cv_splits": CV_SPLITS,
        "n_iter": N_ITER,
        "cv_best_score": row["cv_best_score"],
        "threshold_05": THRESHOLD,
        "best_threshold_f1": best_thr,
        "metrics_05": payload["metrics_05"],
        "metrics_best_f1": payload["metrics_best"],
        "best_params": payload["search"].best_params_,
        "model_path": str(model_path),
        "predictions_path": str(pred_path),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    registry_rows.append(
        {
            "run_name": run_name,
            "model_path": str(model_path),
            "predictions_path": str(pred_path),
            "meta_path": str(meta_path),
        }
    )

registry_df = pd.DataFrame(registry_rows)
summary_df = results_df.merge(registry_df, on="run_name", how="left")
summary_df

In [ ]:
best_idx = int(results_df["f1_best"].idxmax())
best_run = results_df.loc[best_idx, "run_name"]
best_params = results_df.loc[best_idx, "best_params"]

print("Best run (F1 best threshold):", best_run)
print("Best params:")
print(best_params)

## Inclusion MLflow (desactivee par defaut)

Ce bloc est pret pour la prochaine etape.
Aucune execution MLflow tant que `ENABLE_MLFLOW = False`.

In [ ]:
ENABLE_MLFLOW = True
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "accidentologie_model_benchmark"
ENABLE_MODEL_REGISTRY = True
REGISTERED_MODEL_NAME = "briefml-xgboost-product15-v2-time-bucket"

if ENABLE_MLFLOW:
    import numpy as np
    import mlflow
    import mlflow.sklearn
    from mlflow.tracking import MlflowClient

    def _to_params(d):
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(v, (int, float, str, bool, np.integer, np.floating, np.bool_)):
                    out[k] = v.item() if hasattr(v, "item") else v
        return out

    def _normalize_metrics(d, prefix="valid_"):
        keys = {"accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold"}
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if k in keys and isinstance(v, (int, float, np.integer, np.floating)):
                    out[f"{prefix}{k}"] = float(v)
        return out

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

    # On enregistre le meilleur run (par f1_best) dans le registry
    best_run_name = results_df.loc[results_df["f1_best"].idxmax(), "run_name"]

    for row in rows:
        run_name = row["run_name"]
        payload = trained.get(run_name, {})
        search = payload.get("search")
        model = payload.get("model")

        if search is None or model is None:
            print(f"[mlflow] skip {run_name}: missing fitted search/model")
            continue

        metrics_05 = _normalize_metrics(payload.get("metrics_05"), prefix="valid_")
        metrics_best = _normalize_metrics(payload.get("metrics_best"), prefix="valid_bestf1_")

        model_path = ARTIFACT_DIR / f"{run_name}.joblib"
        pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
        meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

        # Enregistrer dans le registry uniquement le meilleur run
        register_name = REGISTERED_MODEL_NAME if (ENABLE_MODEL_REGISTRY and run_name == best_run_name) else None

        with mlflow.start_run(run_name=run_name):
            mlflow.set_tags({
                "notebook": "14_xgboost_optimization.ipynb",
                "model_family": "xgboost",
                "model_flavor": "sklearn",
                "tag": "accidentologie",
                "optimized_for": str(row.get("optimized_for", "unknown")),
            })
            if register_name:
                mlflow.set_tag("registry_enabled", "true")

            mlflow.log_param("seed", int(SEED))
            mlflow.log_param("cv_splits", int(CV_SPLITS))
            mlflow.log_param("n_iter", int(N_ITER))
            mlflow.log_param("target", TARGET)
            mlflow.log_param("cv_primary_metric", str(row.get("optimized_for", "unknown")))
            mlflow.log_params(_to_params(search.best_params_))
            mlflow.log_metric("cv_primary_score", float(search.best_score_))

            if metrics_05:
                mlflow.log_metrics(metrics_05)
            if metrics_best:
                mlflow.log_metrics(metrics_best)

            model_info = mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                registered_model_name=register_name,
            )

            for p, art in [(model_path, "models"), (pred_path, "predictions"), (meta_path, "metadata")]:
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path=art)

            # --- Tags de performance sur le Model Registry ---
            if register_name and getattr(model_info, "registered_model_version", None):
                version = model_info.registered_model_version
                perf_tags = {
                    "f1_best": f"{payload['metrics_best']['f1']:.4f}",
                    "roc_auc": f"{payload['metrics_05']['roc_auc']:.4f}",
                    "recall_best": f"{payload['metrics_best']['recall']:.4f}",
                    "precision_best": f"{payload['metrics_best']['precision']:.4f}",
                    "best_threshold": f"{payload['best_threshold']:.2f}",
                    "optimized_for": str(row.get("optimized_for", "unknown")),
                }
                for tag_key, tag_val in perf_tags.items():
                    client.set_model_version_tag(register_name, version, tag_key, tag_val)
                print(f"[mlflow] model registered: {register_name} v{version}")
                print(f"[mlflow] registry tags: {perf_tags}")

            print(f"[mlflow] logged: {run_name}")
else:
    print("MLflow desactive. Passe ENABLE_MLFLOW=True pour logger les runs XGBoost.")